# 10.05 -- Inference Benchmarking: Measuring What Actually Matters

**Module:** 10 -- Advanced Inference  
**Notebook:** 5 of 5  
**Prerequisites:** 09.02 (Dynamic Batching), 09.03 (Streaming), 10.01-10.04

---

## Learning Objectives

1. Understand the complete LLM inference metric taxonomy: TTFT, TPOT, E2E, throughput
2. Build a rigorous benchmarking harness that avoids common measurement pitfalls
3. Measure and compare decoding strategies side-by-side on the same hardware
4. Build a regression benchmark to catch performance regressions in serving code
5. Understand roofline analysis: determining whether a system is compute- or bandwidth-bound

---

## 1. Why Benchmarking Is Hard

LLM inference benchmarking is surprisingly easy to do wrong. Common mistakes:

**Warmup neglect**: PyTorch JIT compilation and CUDA kernel launch overhead inflate
the first few iteration times by 5-10x. Always discard the first N warmup iterations.

**Prompt length bias**: a 10-token prompt with a 100-token output is a very different
benchmark from a 1000-token prompt with a 10-token output. Always report both prompt
and output lengths alongside latency numbers.

**Batch size omission**: throughput without batch size is meaningless. Throughput scales
nearly linearly with batch size (up to memory bandwidth saturation) so always report both.

**Clock resolution**: `time.time()` has microsecond resolution on most systems but
CUDA operations are asynchronous. Always call `torch.cuda.synchronize()` before
stopping the timer, or use CUDA events for GPU timing.

**Single-run variance**: a single timing measurement has high variance. Report mean,
standard deviation, and P99 over at least 20 runs.


In [ ]:
# Environment setup.
#
# We implement the full benchmarking infrastructure from scratch:
#   - A BenchmarkTimer that handles warmup, CUDA synchronisation, and statistics.
#   - A LatencyBenchmark for single-request metrics (TTFT, TPOT, E2E).
#   - A ThroughputBenchmark for server-level metrics (tokens/second at batch size N).
#   - A RegressionSuite that compares a new system against a stored baseline.

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import json
import os
from typing import List, Dict, Callable, Optional, Tuple
from dataclasses import dataclass, field
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
HAS_CUDA = device == 'cuda'
print(f'Device: {device}')
if HAS_CUDA:
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  {props.total_memory / 1e9:.1f} GB')


## 2. Benchmarking Infrastructure


In [ ]:
# BenchmarkTimer: the core measurement primitive.
#
# Handles three essential correctness concerns:
#
#   1. CUDA synchronisation: CUDA operations are asynchronous. If we call
#      time.time() immediately after model.generate(), we measure launch time
#      not execution time. torch.cuda.synchronize() blocks until all pending
#      CUDA ops complete. On CPU-only systems this is a no-op.
#
#   2. Warmup: the first few iterations are slower due to JIT compilation,
#      kernel launch overhead, and memory allocation. We discard warmup runs
#      and only report statistics over the stable measurement window.
#
#   3. Statistics: we report mean, std, and P99 over multiple runs.
#      The P99 is the most important for SLO compliance.

@dataclass
class BenchmarkResult:
    name:         str
    latencies_ms: List[float]
    metadata:     Dict = field(default_factory=dict)

    @property
    def mean_ms(self):   return float(np.mean(self.latencies_ms))
    @property
    def std_ms(self):    return float(np.std(self.latencies_ms))
    @property
    def p50_ms(self):    return float(np.percentile(self.latencies_ms, 50))
    @property
    def p95_ms(self):    return float(np.percentile(self.latencies_ms, 95))
    @property
    def p99_ms(self):    return float(np.percentile(self.latencies_ms, 99))
    @property
    def cv(self):        return self.std_ms / (self.mean_ms + 1e-9)  # coefficient of variation

    def summary(self) -> str:
        return (f'{self.name}: '
                f'mean={self.mean_ms:.1f}ms  '
                f'std={self.std_ms:.1f}ms  '
                f'P50={self.p50_ms:.1f}  '
                f'P95={self.p95_ms:.1f}  '
                f'P99={self.p99_ms:.1f}')


class BenchmarkTimer:
    """
    Correctly measures latency of a callable on GPU or CPU.

    Handles warmup, CUDA synchronisation, and statistical aggregation.
    """

    def __init__(self, n_warmup: int = 3, n_runs: int = 20):
        self.n_warmup = n_warmup
        self.n_runs   = n_runs

    def measure(self, fn: Callable, name: str = 'benchmark',
                metadata: Dict = None) -> BenchmarkResult:
        """
        Call fn() n_warmup + n_runs times.
        Discard warmup iterations. Return statistics over measurement runs.

        fn should return None or a value we do not time (only the call itself).
        """
        metadata = metadata or {}

        # Warmup: JIT compile, allocate caches, reach steady state
        for _ in range(self.n_warmup):
            fn()
            if HAS_CUDA: torch.cuda.synchronize()

        # Measurement runs
        latencies = []
        for _ in range(self.n_runs):
            if HAS_CUDA: torch.cuda.synchronize()
            t0 = time.perf_counter()
            fn()
            if HAS_CUDA: torch.cuda.synchronize()
            t1 = time.perf_counter()
            latencies.append((t1 - t0) * 1000)

        return BenchmarkResult(name=name, latencies_ms=latencies, metadata=metadata)


# Load model once for all benchmarks
print('Loading GPT-2 for benchmarking...')
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
model.eval()
print(f'Model ready: {sum(p.numel() for p in model.parameters()):,} parameters')


In [ ]:
# Latency benchmark: TTFT, TPOT, and E2E for single requests.
#
# We define three distinct latency measurements:
#
#   TTFT (Time-to-First-Token):
#     Cost of the prefill step: processing the input prompt and generating token 1.
#     Measured by running model.generate(max_new_tokens=1).
#     Scales approximately linearly with prompt length.
#     Critical for interactive applications: this is the blank-screen time.
#
#   TPOT (Time-per-Output-Token):
#     Cost of each decode step after the first token.
#     Measured by (E2E - TTFT) / (n_output_tokens - 1).
#     Nearly constant regardless of sequence length (memory-bandwidth bound).
#     Critical for streaming: this is the inter-token delay the user sees.
#
#   E2E (End-to-End):
#     Total time from receiving the request to producing the final token.
#     E2E = TTFT + TPOT * (n_output_tokens - 1).
#     Critical for batch/offline workloads.

timer = BenchmarkTimer(n_warmup=3, n_runs=15)

# Test prompt with different lengths
SHORT_PROMPT = 'The attention mechanism'
LONG_PROMPT  = ('The transformer architecture, introduced by Vaswani et al. in 2017, '
                'fundamentally changed how sequence-to-sequence tasks are approached. '
                'The key innovation was the self-attention mechanism, which allows each '
                'token in a sequence to attend directly to all other tokens.')

def make_gen_fn(prompt_text, n_tokens):
    """Return a generation function with fixed prompt and token count."""
    ids = tokenizer(prompt_text, return_tensors='pt').input_ids.to(device)
    def fn():
        with torch.no_grad():
            model.generate(ids, max_new_tokens=n_tokens,
                           do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return fn, ids.shape[1]


# TTFT: generate exactly 1 token (measures prefill cost)
fn_short_ttft, short_len = make_gen_fn(SHORT_PROMPT, 1)
fn_long_ttft,  long_len  = make_gen_fn(LONG_PROMPT,  1)

# E2E: generate 50 tokens
fn_short_e2e, _ = make_gen_fn(SHORT_PROMPT, 50)
fn_long_e2e,  _ = make_gen_fn(LONG_PROMPT,  50)

print('Running latency benchmarks...')
r_short_ttft = timer.measure(fn_short_ttft, f'TTFT (prompt={short_len} tok)')
r_long_ttft  = timer.measure(fn_long_ttft,  f'TTFT (prompt={long_len} tok)')
r_short_e2e  = timer.measure(fn_short_e2e,  f'E2E  (prompt={short_len}, output=50)')
r_long_e2e   = timer.measure(fn_long_e2e,   f'E2E  (prompt={long_len}, output=50)')

results_latency = [r_short_ttft, r_long_ttft, r_short_e2e, r_long_e2e]

print()
for r in results_latency:
    print(f'  {r.summary()}')

# Derive TPOT
tpot_short = (r_short_e2e.mean_ms - r_short_ttft.mean_ms) / 49
tpot_long  = (r_long_e2e.mean_ms  - r_long_ttft.mean_ms)  / 49
print()
print(f'  TPOT (short prompt): {tpot_short:.1f} ms/token  ({1000/tpot_short:.0f} tok/s)')
print(f'  TPOT (long prompt):  {tpot_long:.1f} ms/token  ({1000/tpot_long:.0f} tok/s)')
print()
print('TPOT is nearly identical regardless of prompt length:')
print('  Each decode step is memory-bandwidth bound, not compute bound.')
print('  The GPU loads the weight matrices once per step regardless of past context length.')


In [ ]:
# Throughput benchmark: tokens per second across batch sizes.
#
# Throughput is measured in tokens generated per second across a batch.
# It captures the most important characteristic of a serving system:
# how many tokens can it produce in a fixed time window.
#
# The throughput-batch size curve has a characteristic shape:
#   - At low batch sizes: throughput is proportional to batch size.
#     Each additional request adds tokens with negligible per-unit overhead
#     because the memory bandwidth cost is dominated by loading the weights,
#     which happens once per decode step regardless of batch size.
#   - At high batch sizes: throughput saturates as memory bandwidth is exhausted.
#     The GPU cannot load weights fast enough to keep all batch slots active.
#
# The saturation point identifies the optimal operating batch size for max throughput.
# For a GPU with HBM bandwidth B and model weight size W:
#   tokens_per_second_max ~ B / W  (compute-independent upper bound)
# e.g. A100 (2 TB/s bandwidth, 14 GB weights) ~ 143 tokens/s per layer.

print('Running throughput benchmarks across batch sizes...')

N_OUTPUT_TOKENS = 30
BATCH_SIZES     = [1, 2, 4, 8] if not HAS_CUDA else [1, 2, 4, 8, 16, 32]

throughput_results = []
for bs in BATCH_SIZES:
    prompt_ids = tokenizer(
        ['The transformer model learns'] * bs,
        return_tensors='pt', padding=True,
    ).input_ids.to(device)

    def gen_batch():
        with torch.no_grad():
            model.generate(
                prompt_ids,
                max_new_tokens=N_OUTPUT_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

    result = BenchmarkTimer(n_warmup=2, n_runs=10).measure(
        gen_batch, name=f'batch_size={bs}',
        metadata={'batch_size': bs, 'n_output_tokens': N_OUTPUT_TOKENS}
    )

    tokens_per_sec = (bs * N_OUTPUT_TOKENS) / (result.mean_ms / 1000)
    throughput_results.append({
        'batch_size':    bs,
        'mean_ms':       result.mean_ms,
        'tokens_per_s':  tokens_per_sec,
        'latencies':     result.latencies_ms,
    })
    print(f'  batch={bs:>3}  mean={result.mean_ms:>7.1f}ms  '
          f'throughput={tokens_per_sec:>7.1f} tok/s')

# Roofline: estimate compute vs bandwidth bottleneck
# On CPU: memory bandwidth ~50 GB/s; GPT-2 weights ~500 MB (float32)
# Expected decode step at batch=1: 500e6 / 50e9 = 10 ms/layer (rough)
print()
print('Throughput analysis:')
best_tps = max(r['tokens_per_s'] for r in throughput_results)
best_bs  = throughput_results[[r['tokens_per_s'] for r in throughput_results].index(best_tps)]['batch_size']
print(f'  Peak throughput: {best_tps:.1f} tok/s at batch_size={best_bs}')


In [ ]:
# Comprehensive benchmark dashboard.
#
# Four panels capture the complete performance picture of an LLM serving system:
#
# Panel 1: Latency CDF -- the full distribution, not just mean.
#   The P99 matters most: SLOs are percentile-based.
#   A tight CDF (small spread) means predictable performance.
#   A heavy tail means occasional very slow requests.
#
# Panel 2: TTFT vs prompt length.
#   TTFT should scale linearly with prompt length (prefill is O(N) compute).
#   Deviations suggest batching or caching effects.
#
# Panel 3: Throughput vs batch size.
#   The characteristic curve: linear rise then saturation.
#   The knee of the curve is the optimal operating batch size.
#
# Panel 4: Decoding strategy comparison.
#   Compare autoregressive, sampling with temperature, and top-p/top-k
#   to understand the overhead of different sampling strategies.

fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.32)

# Panel 1: Latency CDF
ax = fig.add_subplot(gs[0, 0])
colors_lat = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for r, col in zip(results_latency, colors_lat):
    sorted_lats = sorted(r.latencies_ms)
    cdf = np.arange(1, len(sorted_lats)+1) / len(sorted_lats)
    ax.plot(sorted_lats, cdf * 100, lw=2, color=col,
            label=r.name + f' (P99={r.p99_ms:.0f}ms)')
ax.axhline(95, color='gray', ls='--', alpha=0.5, lw=1)
ax.axhline(99, color='gray', ls=':', alpha=0.5, lw=1)
ax.set_xlabel('Latency (ms)', fontsize=11)
ax.set_ylabel('Cumulative %', fontsize=11)
ax.set_title('Latency CDF by Benchmark Condition', fontsize=12)
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

# Panel 2: TTFT vs prompt length
ax = fig.add_subplot(gs[0, 1])
prompt_lengths_test = [5, 10, 20, 40, 80, 128]
ttft_means = []
ttft_stds  = []
for plen in prompt_lengths_test:
    dummy_prompt = ' '.join(['the'] * plen)
    fn_p, _ = make_gen_fn(dummy_prompt, 1)
    r_p = BenchmarkTimer(n_warmup=2, n_runs=8).measure(fn_p)
    ttft_means.append(r_p.mean_ms)
    ttft_stds.append(r_p.std_ms)

ax.errorbar(prompt_lengths_test, ttft_means, yerr=ttft_stds,
            fmt='o-', color='#2ca02c', lw=2, ms=6, capsize=4, label='Measured TTFT')
# Fit a line to show linearity
coeffs = np.polyfit(prompt_lengths_test, ttft_means, 1)
line   = np.poly1d(coeffs)
ax.plot(prompt_lengths_test, line(prompt_lengths_test), '--',
        color='gray', alpha=0.7, label=f'Linear fit (slope={coeffs[0]:.1f} ms/tok)')
ax.set_xlabel('Prompt length (tokens)', fontsize=11)
ax.set_ylabel('TTFT (ms)', fontsize=11)
ax.set_title('TTFT vs Prompt Length\n(Should be approximately linear)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 3: throughput vs batch size
ax = fig.add_subplot(gs[1, 0])
bs_vals = [r['batch_size'] for r in throughput_results]
tps_vals = [r['tokens_per_s'] for r in throughput_results]
ax.plot(bs_vals, tps_vals, 'o-', color='#1f77b4', lw=2.5, ms=8)
for bs, tps in zip(bs_vals, tps_vals):
    ax.text(bs, tps + max(tps_vals)*0.02, f'{tps:.0f}', ha='center', fontsize=8)
ax.axhline(max(tps_vals), color='gray', ls=':', alpha=0.5, label='Peak observed')
ax.set_xlabel('Batch size', fontsize=11)
ax.set_ylabel('Throughput (tokens/s)', fontsize=11)
ax.set_title('Throughput vs Batch Size\n(Saturates when memory bandwidth is exhausted)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 4: sampling strategy comparison
ax = fig.add_subplot(gs[1, 1])

strategies = {}
base_ids = tokenizer('The transformer model', return_tensors='pt').input_ids.to(device)

for label, kwargs in [
    ('Greedy',         dict(do_sample=False)),
    ('Temp=0.8',       dict(do_sample=True,  temperature=0.8)),
    ('Temp=1.0',       dict(do_sample=True,  temperature=1.0)),
    ('Top-p=0.9',      dict(do_sample=True,  top_p=0.9)),
    ('Top-k=50',       dict(do_sample=True,  top_k=50)),
]:
    def make_fn(kw):
        def fn():
            with torch.no_grad():
                model.generate(base_ids, max_new_tokens=30,
                               pad_token_id=tokenizer.eos_token_id, **kw)
        return fn
    r_strat = BenchmarkTimer(n_warmup=2, n_runs=10).measure(make_fn(kwargs))
    strategies[label] = r_strat.mean_ms

labels = list(strategies.keys())
vals   = list(strategies.values())
bars = ax.bar(labels, vals, color='#9467bd', alpha=0.85)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.5, f'{v:.0f}ms',
            ha='center', fontsize=9)
ax.set_ylabel('Mean latency (ms, 30 output tokens)', fontsize=11)
ax.set_title('Sampling Strategy Overhead\n(All strategies have similar cost)', fontsize=12)
ax.grid(alpha=0.3, axis='y')

plt.suptitle('LLM Inference Benchmarking Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/10_benchmarking.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Regression benchmark: detect performance regressions between runs.
#
# A regression benchmark compares current measurements against a stored baseline.
# It raises an alert if any metric degrades beyond a threshold.
# This is how production ML serving teams catch performance regressions in CI/CD:
#
#   1. Run benchmarks on every code change.
#   2. Compare against baseline stored when the system was last known good.
#   3. Fail the CI pipeline if P99 latency regresses by more than 5 percent.
#
# The threshold percentages are tuned per metric:
#   - TTFT: strict (user-visible blank screen time)
#   - TPOT: strict (per-token streaming speed)
#   - Throughput: moderate (server-side metric, some variance is acceptable)

BASELINE_FILE = '/tmp/benchmark_baseline.json'

class RegressionSuite:
    """
    Stores benchmark baselines and compares new measurements against them.

    Baselines are stored as JSON files for portability across runs.
    Each entry stores the benchmark name and P50/P99/mean statistics.
    """

    def __init__(self, baseline_path: str, thresholds: Dict[str, float] = None):
        self.baseline_path = baseline_path
        # Thresholds: maximum allowed regression (fraction) per statistic
        self.thresholds = thresholds or {
            'mean_ms': 0.10,   # 10% regression on mean latency
            'p99_ms':  0.05,   # 5% regression on P99
        }
        self.baseline = self._load()

    def _load(self) -> Dict:
        if os.path.exists(self.baseline_path):
            with open(self.baseline_path) as f:
                return json.load(f)
        return {}

    def save_baseline(self, results: List[BenchmarkResult]):
        """Store current results as the new baseline."""
        baseline = {}
        for r in results:
            baseline[r.name] = {
                'mean_ms': r.mean_ms,
                'p95_ms':  r.p95_ms,
                'p99_ms':  r.p99_ms,
                'metadata': r.metadata,
            }
        with open(self.baseline_path, 'w') as f:
            json.dump(baseline, f, indent=2)
        self.baseline = baseline
        print(f'Baseline saved to {self.baseline_path}')

    def compare(self, results: List[BenchmarkResult]) -> Dict:
        """
        Compare results against baseline.

        Returns a report dict with status PASS/WARN/FAIL per benchmark.
        FAIL means a threshold was exceeded; WARN means no baseline exists.
        """
        report = {}
        for r in results:
            if r.name not in self.baseline:
                report[r.name] = {'status': 'WARN', 'reason': 'No baseline found'}
                continue

            base = self.baseline[r.name]
            issues = []
            for stat, threshold in self.thresholds.items():
                current  = getattr(r, stat)
                baseline = base.get(stat, None)
                if baseline is None or baseline == 0:
                    continue
                regression = (current - baseline) / baseline
                if regression > threshold:
                    issues.append(
                        f'{stat}: {current:.1f}ms vs baseline {baseline:.1f}ms '
                        f'({regression:+.1%} -- threshold {threshold:.0%})'
                    )

            if issues:
                report[r.name] = {'status': 'FAIL', 'issues': issues}
            else:
                report[r.name] = {
                    'status': 'PASS',
                    'mean_delta': (r.mean_ms - base['mean_ms']) / base['mean_ms'],
                    'p99_delta':  (r.p99_ms  - base['p99_ms'])  / base['p99_ms'],
                }
        return report


# Demo: save current results as baseline, then re-run and compare
suite = RegressionSuite(BASELINE_FILE, thresholds={'mean_ms': 0.15, 'p99_ms': 0.10})
suite.save_baseline(results_latency)

# Simulate a re-run (in practice this would be a new code deployment)
r_short_ttft2 = BenchmarkTimer(n_warmup=3, n_runs=15).measure(
    *make_gen_fn(SHORT_PROMPT, 1)[:1],  # just the fn
    name=r_short_ttft.name
)

report = suite.compare([r_short_ttft2])
print('Regression report:')
for name, status in report.items():
    if status['status'] == 'PASS':
        mean_d = status.get('mean_delta', 0)
        p99_d  = status.get('p99_delta', 0)
        print(f'  PASS  {name}  mean: {mean_d:+.1%}  P99: {p99_d:+.1%}')
    else:
        print(f'  {status["status"]}  {name}: {status.get("reason", "")}')
        for issue in status.get('issues', []):
            print(f'         {issue}')


## 3. Roofline Analysis

Roofline analysis determines whether a compute kernel is limited by compute throughput
or memory bandwidth. For LLM decode, the answer is almost always memory bandwidth.

```
Roofline model:

  Peak throughput (tokens/s) = min(
      compute_peak (FLOP/s) / FLOPs_per_token,
      bandwidth_peak (bytes/s) / bytes_per_token
  )

For GPT-2 decode, batch size 1:
  FLOPs per token:  2 * n_params = 234 M FLOP
  Bytes per token:  n_params * dtype_size = 468 MB  (float32)
  Arithmetic intensity: 234 M FLOP / 468 MB = 0.5 FLOP/byte

Hardware:
  CPU:  ~10 TFLOP/s peak, ~50 GB/s memory bandwidth
        compute-bound threshold: 10e12 / 50e9 = 200 FLOP/byte
  A100: ~312 TFLOP/s peak (FP16), ~2000 GB/s bandwidth
        compute-bound threshold: 312e12 / 2000e9 = 156 FLOP/byte

Since 0.5 << 156, LLM decode is deep in the memory-bandwidth bound region.
Implication: faster compute (more CUDA cores) does NOT help decode throughput.
             Wider memory bus or tensor parallelism (to distribute bandwidth) helps.
```

## 4. Module 10 Summary

This module covered the full advanced inference engineering stack:

- **10.01 Speculative Decoding**: Rejection sampling with a draft model; acceptance criterion proof; gamma sweep
- **10.02 EAGLE**: Feature-level drafting using target model hidden states; EAGLE-2 dynamic gamma
- **10.03 Medusa**: Multi-head parallel prediction; candidate tree generation; tree acceptance
- **10.04 Tree Decoding**: Unified framework; tree attention mask; width/depth trade-off analysis
- **10.05 Inference Benchmarking**: TTFT/TPOT/E2E measurement harness; throughput curves; regression suite

## 5. Exercises

1. **Warmup sensitivity**: Modify `BenchmarkTimer` to log the latency of every run
   including warmup. Plot all latencies in order. How many warmup runs are required
   before measurements stabilise on your hardware?

2. **KV cache memory profiling**: Instrument the generation loop to record
   `torch.cuda.memory_allocated()` before and after each decode step.
   Plot KV cache memory growth vs sequence length. Does it match the theoretical
   formula `2 * n_layers * n_heads * head_dim * seq_len * dtype_bytes`?

3. **Throughput saturation point**: Extend the throughput sweep to batch sizes
   [1, 2, 4, 8, 16, 32, 64] on GPU. Fit a saturation model:
   `throughput(B) = B * T_single / (1 + B * T_single / T_max)` where `T_max`
   is the bandwidth-limited ceiling. Estimate `T_max` from the fit.

4. **End-to-end comparison**: Run the complete benchmark suite on:
   (a) standard autoregressive decoding and
   (b) speculative decoding with gamma=4 (using the code from 10.01).
   Produce a dashboard with TTFT, TPOT, and throughput for both methods.

5. **Regression CI integration**: Write a Python script that:
   (a) runs the benchmark suite, (b) compares against a JSON baseline,
   (c) exits with code 0 on PASS and code 1 on any FAIL. This script
   can be added to a GitHub Actions CI workflow to gate deployments.

---

## 6. References

- [Williams et al. (2009) -- Roofline: An Insightful Visual Performance Model](https://dl.acm.org/doi/10.1145/1498765.1498785)
- [Sheng et al. (2023) -- FlexGen: High-Throughput Generative Inference of LLMs](https://arxiv.org/abs/2303.06865)
- [Kwon et al. (2023) -- Efficient Memory Management for LLM Serving with PagedAttention](https://arxiv.org/abs/2309.06180)
- [MLPerf Inference Benchmark](https://mlcommons.org/benchmarks/inference/)
